# Aggregate TELL Load Data by NERC TPL-08 Region

This notebook takes an existing set of simulations using the Total ELectricity Loads (TELL) model and aggregates the BA-level data within each NERC TPL-08 region. The original TELL simulations were conducted using code stored here: https://github.com/cdburley/foresight_energy_drought_evolution_loads. We are using the 2024 runs which passes 45 years (1980-2024) of weather through the ML-based load models trained on data from 2021-2023. The loads are scaled to match the 2024 annual energy values from the IM3 rcp45hotter_ssp3 GCAM-USA run which tracks reasonably well with load growth over the last few years. All years have the same annual energy but different hour-to-hour variability.

The absolute values of the loads should be taken with a grain of salt. TELL BAs also do not directly align with the NERC TPL-08 regions. Notably, TELL models MISO and SPP as single BAs whereas they're split up in the TPL-08 shapefiles. For lack of more information I just took the MISO loads and split them in half to create hourly loads for MISO-N and MISO-S. Same for SPP. Annual peak loads by interconnection are in the ballpark of publicly available information on the peaks.


In [ ]:
# Start by importing the packages we need:
import os
import datetime
import warnings

import pandas as pd
import numpy as np


## Suppress Future Warnings


In [ ]:
warnings.simplefilter(action='ignore', category=FutureWarning)


## Set the Directory Structure

In [ ]:
# Identify the data input and directories:
tell_load_data_dir =  '/Users/burl878/Documents/Code/code_repos/foresight_energy_drought_evolution_loads/data/tell_output/rcp45hotter_ssp3/2024/'
ba_mapping_data_dir = '/Users/burl878/Documents/Code/code_repos/gdo_climate_toolsuite_visualizations/data/'
data_output_dir = '/Users/burl878/Documents/Code/code_repos/gdo_climate_toolsuite_visualizations/data/load_data/'


## Create a Function to Aggregate TELL BA-Level Data for Each NERC TPL-08 Region


In [ ]:
def aggregate_tell_load_data(start_year: int, end_year: int, tell_load_data_dir: str, ba_mapping_data_dir: str, data_output_dir: str):

    # Read in TELL BA to NERC TPL-08 region mapping:
    mapping = pd.read_csv((ba_mapping_data_dir + 'nerc_tpl08_region_to_TELL_BA_mapping.csv'))
    
    # Extract the list of NERC TPL-08 region names:
    nerc_tpl08_regions = mapping['short_name'].unique()
                
    # Loop over the years of TELL load data:
    for year in range(start_year, end_year, 1):
        # Read in the raw TELL BA output file for the TELL weather year:
        ba_df = pd.read_csv((tell_load_data_dir + 'TELL_Balancing_Authority_Hourly_Load_Data_' + str(year) + '_Scaled_2024.csv'))

        # Loop over each of the NERC TPL-08 regions:
        for i in range(len(nerc_tpl08_regions)):
        
            # Make a list of the BAs that map to that NERC TPL-08 region:
            tell_subset = mapping[(mapping['short_name'] == nerc_tpl08_regions[i])].copy()
            tell_bas = tell_subset['tell_ba'].unique()
            
            # Subset the TELL output file to just the BAs in that NERC TPL-08 region:
            ba_subset_df = ba_df[ba_df['BA_Code'].isin(tell_bas)].copy()

            # Sum the TELL BA-level loads across all BAs in that NERC TPL-08 region:
            ba_subset_df['Region_Load_MWh'] = ba_subset_df.groupby('Time_UTC')['Scaled_TELL_BA_Load_MWh'].transform('sum').round(2)

            # Only keep the variable we need, drop duplicates, and sort chronologically:
            region_df = ba_subset_df[['Time_UTC','Region_Load_MWh']]
            region_df = region_df.drop_duplicates().sort_values('Time_UTC')
            region_df.reset_index(inplace=True, drop=True)
            
            # Rename the regional load to the region name:
            region_df.rename(columns={'Region_Load_MWh': nerc_tpl08_regions[i]}, inplace=True)
        
            # Aggregate the output into a new dataframe:
            if i == 0:
               year_df = region_df
            else:
               year_df = year_df.merge(region_df, on=['Time_UTC'])

            # Clean up and move to the next region:
            del tell_subset, tell_bas, ba_subset_df, region_df
        
        # Aggregate the output into a new dataframe:
        if year == start_year:
           output_df = year_df
        else:
           output_df = pd.concat([output_df, year_df])

        # Clean up and move to the next year:
        del ba_df, i, year_df

    # Sum up the regions in each interconnection:
    output_df['WECC'] = (output_df['CA'] + output_df['PNW'] + output_df['GB'] + output_df['SW'] + output_df['RM']).round(2)
    output_df['EIC'] = (output_df['SPP'] + output_df['MISO'] + output_df['SERC'] + output_df['FL'] + output_df['PJM'] + output_df['NYISO'] + output_df['ISONE']).round(2)

    # Split MISO and SPP into north and south components using a 50/50 split:
    output_df['MISO-N'] = (output_df['MISO'] * 0.5).round(2)
    output_df['MISO-S'] = (output_df['MISO'] * 0.5).round(2)
    output_df['SPP-N'] = (output_df['SPP'] * 0.5).round(2)
    output_df['SPP-S'] = (output_df['SPP'] * 0.5).round(2)
    
    # Only keep the columns we need:
    output_df = output_df[['Time_UTC', 'EIC', 'ERCOT', 'WECC', 'CA', 'FL', 'GB', 'ISONE', 'MISO-N', 'MISO-S', 'NYISO', 'PJM', 'PNW', 'RM', 'SERC', 'SPP-N', 'SPP-S', 'SW']].copy()
    
    # Reset the index value:
    output_df.reset_index(inplace=True, drop=True)
    
    # Set the output filename:
    output_filename = ('NERC_Region_Hourly_Loads_' + str(start_year) + '_to_' + str((end_year-1)) + '.csv')
        
    # Write out the dataframe to a .csv file:
    output_df.to_csv((os.path.join(data_output_dir, output_filename)), sep=',', index=False)
    
    # Return the output dataframe:
    return output_df


In [ ]:
# Execute the function:
output_df = aggregate_tell_load_data(start_year = 1980,
                                     end_year = 2025, 
                                     tell_load_data_dir = tell_load_data_dir,
                                     ba_mapping_data_dir = ba_mapping_data_dir,
                                     data_output_dir = data_output_dir)

output_df


## Create a Function to Calculate the Peak Load by Day


In [ ]:
def process_daily_data(start_year: int, end_year: int, data_output_dir: str):

    # Read in data processed using the function created above:
    load_df = pd.read_csv((data_output_dir + 'NERC_Region_Hourly_Loads_' + str(start_year) + '_to_' + str((end_year-1)) + '.csv'))

    # Set the time as datetime variable:
    load_df['Time_UTC'] = pd.to_datetime(load_df['Time_UTC'])

    # Extract the day from the datetime variable:
    load_df['Date'] = load_df['Time_UTC'].dt.date

    # Compute the daily max for each region:
    load_df['EIC_Max'] = load_df.groupby('Date')['EIC'].transform('max').round(2)
    load_df['ERCOT_Max'] = load_df.groupby('Date')['ERCOT'].transform('max').round(2)
    load_df['WECC_Max'] = load_df.groupby('Date')['WECC'].transform('max').round(2)
    load_df['CA_Max'] = load_df.groupby('Date')['CA'].transform('max').round(2)
    load_df['FL_Max'] = load_df.groupby('Date')['FL'].transform('max').round(2)
    load_df['GB_Max'] = load_df.groupby('Date')['GB'].transform('max').round(2)
    load_df['ISONE_Max'] = load_df.groupby('Date')['ISONE'].transform('max').round(2)
    load_df['MISO-N_Max'] = load_df.groupby('Date')['MISO-N'].transform('max').round(2)
    load_df['MISO-S_Max'] = load_df.groupby('Date')['MISO-S'].transform('max').round(2)
    load_df['NYISO_Max'] = load_df.groupby('Date')['NYISO'].transform('max').round(2)
    load_df['PJM_Max'] = load_df.groupby('Date')['PJM'].transform('max').round(2)
    load_df['PNW_Max'] = load_df.groupby('Date')['PNW'].transform('max').round(2)
    load_df['RM_Max'] = load_df.groupby('Date')['RM'].transform('max').round(2)
    load_df['SERC_Max'] = load_df.groupby('Date')['SERC'].transform('max').round(2)
    load_df['SPP-N_Max'] = load_df.groupby('Date')['SPP-N'].transform('max').round(2)
    load_df['SPP-S_Max'] = load_df.groupby('Date')['SPP-S'].transform('max').round(2)
    load_df['SW_Max'] = load_df.groupby('Date')['SW'].transform('max').round(2)
    
    # Subset and reorder the columns, drop duplicates, and sort chronologically:
    output_df = load_df[['Date', 'EIC_Max', 'ERCOT_Max', 'WECC_Max', 'CA_Max', 'FL_Max', 'GB_Max', 'ISONE_Max', 'MISO-N_Max', 'MISO-S_Max', 'NYISO_Max',
                         'PJM_Max', 'PNW_Max', 'RM_Max', 'SERC_Max', 'SPP-N_Max', 'SPP-S_Max', 'SW_Max']]
    output_df = output_df.drop_duplicates()
    output_df = output_df.sort_values('Date')

    # Rename the columns for simplicity:
    output_df.rename(columns={'WECC_Max': 'WECC', 'EIC_Max': 'EIC', 'ERCOT_Max': 'ERCOT', 'CA_Max': 'CA', 'FL_Max': 'FL', 'GB_Max': 'GB', 'ISONE_Max': 'ISONE',
                              'MISO-N_Max': 'MISO-N', 'MISO-S_Max': 'MISO-S', 'NYISO_Max': 'NYISO', 'PJM_Max': 'PJM', 'PNW_Max': 'PNW', 'RM_Max': 'RM', 'SERC_Max': 'SERC',
                              'SPP-N_Max': 'SPP-N', 'SPP-S_Max': 'SPP-S', 'SW_Max': 'SW'}, inplace=True)

    # Replace zeros with NaN and reset the index value:
    output_df.replace(0, np.nan, inplace=True)
    output_df.reset_index(inplace=True, drop=True)
    
    # Set the output filename:
    output_filename = ('NERC_Region_Daily_Peak_Loads_' + str(start_year) + '_to_' + str((end_year-1)) + '.csv')
        
    # Write out the dataframe to a .csv file:
    output_df.to_csv((os.path.join(data_output_dir, output_filename)), sep=',', index=False)
    
    return output_df


In [ ]:
# Execute the function:
output_df = process_daily_data(start_year = 1980,
                               end_year = 2025, 
                               data_output_dir = data_output_dir)

output_df
